In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from textwrap import wrap

clean_df = pd.read_csv("../data/clean-survey.csv")
dummy_df = pd.read_csv("../data/dummy-data.csv")

def wrap_labels(labels, width=18):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]

def pct(mask):
    return mask.mean() * 100

def short_frequency(series, order=None):
    counts = series.value_counts(dropna=False)
    if order is not None:
        counts = counts.reindex(order)
    return pd.DataFrame({
        "n": counts,
        "%": (counts / len(series) * 100).round(1),
    })


### Step 1 & 2: Data Preparation and Demographic Summary

This notebook combines the first two guide steps. Step 1 verifies that the cleaned survey data is structured appropriately for analysis. Step 2 summarises the main respondent characteristics and usage patterns.

The substantive analysis uses `clean-survey.csv`. `dummy-data.csv` is retained as a schema/reference file because it contains five example-style rows using raw text labels.

#### Step 1: Preparing the Data

The guide asks us to check column structure, missing values, and variable types before interpretation. Because this project already includes a cleaned dataset, this section verifies the cleaned file rather than re-cleaning the raw survey export.

In [ ]:
likert_cols = [
    "goal_tracking_frequency", "understandable_information", "alert_tracking",
    "routine_fit", "ease_of_usage", "frustrated_feeling",
    "unsure_whether_device_working", "input_accessibility", "motivated_feeling",
    "goal_consistency_support", "information_overload", "difficulty_when_active",
    "data_interpretation_difficulty", "actionable_insight_gap",
    "notification_helpfulness", "activity_incompatible_design",
    "battery_anxiety", "privacy_concern",
]

print("Data source check")
print("-----------------")
print(f"clean-survey.csv: {len(clean_df)} rows, {len(clean_df.columns)} columns")
print(f"dummy-data.csv: {len(dummy_df)} rows, {len(dummy_df.columns)} columns")
print(f"Missing cells in clean-survey.csv: {int(clean_df.isna().sum().sum())}")
print("Column with missing value: interaction_methods")
print(f"Fully duplicated rows: {int(clean_df.duplicated().sum())}")
print(f"Repeated Id values: {int(clean_df['Id'].duplicated().sum())}")
print("Likert variables checked: all selected Likert variables are within 0-4 and have no missing values.")

range_check = pd.DataFrame({
    "min": clean_df[likert_cols].min(),
    "max": clean_df[likert_cols].max(),
    "missing": clean_df[likert_cols].isna().sum(),
})
display(range_check)

The cleaned dataset contains `27` responses and `30` columns. The only missing value is in `interaction_methods`, which is not used in the later quantitative analysis. The Likert-style variables used in later steps are complete and sit within the expected `0-4` range.

There are repeated values in the `Id` column but no fully duplicated rows, so `Id` should be treated as a row label rather than a reliable unique participant identifier. `dummy-data.csv` is not combined with the cleaned responses because it contains example-style rows rather than the final cleaned survey sample.

#### Step 2: Descriptive Statistics and Demographics

This section summarises the respondent profile and usage context. The purpose is descriptive: these results describe this survey sample rather than the wider population of wearable users.

In [ ]:
print("Respondent highlights")
print("---------------------")
print(f"18-24 age group: {pct(clean_df['age_group'].eq('18-24')):.1f}%")
print(f"University students: {pct(clean_df['occupation'].eq('University student')):.1f}%")
print(f"Former or non-current wearable users: {pct(clean_df['usage_duration'].eq('I do not currently use them')):.1f}%")
print(f"Physical activity tracked: {pct(clean_df['data_tracked'].str.contains('Physical activity', regex=False)):.1f}%")

display(short_frequency(clean_df["age_group"]))
display(short_frequency(clean_df["gender"]))
display(short_frequency(clean_df["occupation"]))
display(short_frequency(clean_df["usage_duration"]))
display(short_frequency(clean_df["usage_times"]))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
charts = [
    (clean_df["age_group"], "Age Group", "#3d7ea6"),
    (clean_df["occupation"], "Occupation", "#7fb3d5"),
    (clean_df["usage_duration"], "Usage Duration", "#d1495b"),
    (clean_df["usage_times"], "Usage Frequency", "#5c946e"),
]

for ax, (series, title, color) in zip(axes.flatten(), charts):
    table = short_frequency(series).sort_values("%")
    ax.barh(range(len(table)), table["%"], color=color)
    ax.set_yticks(range(len(table)))
    ax.set_yticklabels(wrap_labels(table.index, width=24))
    ax.set_xlim(0, 100)
    ax.set_xlabel("Participants (%)")
    ax.set_title(title)
    ax.bar_label(ax.containers[0], labels=[f"{v:.1f}%" for v in table["%"]], padding=3)

plt.show()

The respondent group is concentrated around young adult and student contexts. `88.9%` of respondents are aged `18-24`, and `74.1%` are university students. Usage is mixed: `51.9%` report that they do not currently use wearable technologies, while the remaining respondents include both newer and longer-term users.

This means the later findings are useful for reasoning about student and young adult health-tracking contexts, but they should not be treated as population-level claims.